# F4 · Preise: Wie verteilen sich die Preise in den Kategorien?
**Zuständig:** Tim

Grundlage sind die bereinigten Tabellen (`_cl`). Die Kategorien in diesem Abschnitt sind **vorläufig**, bis Miguels Einteilung aus F3.1 fertig ist.

## Startzelle (gemeinsam)
Diese Zelle steht im Team-Notebook **nur einmal** ganz oben in der Analyse. Wenn sie dort schon steht, diese Zelle **nicht** mitkopieren.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Einheitlicher Stil für alle Diagramme
sns.set_theme(style="whitegrid")

# Bereinigte Tabellen laden
orders_cl = pd.read_csv("orders_cl.csv", parse_dates=["created_date"])
orderlines_cl = pd.read_csv("orderlines_cl.csv", parse_dates=["date"])
products_cl = pd.read_csv("products_cl.csv")
brands = pd.read_csv("brands.csv")

# Arbeitstabelle: jede Zeile = ein verkaufter Artikel
basis = orderlines_cl.copy()
basis["rabatt_eur"] = basis["price"] - basis["unit_price"]          # Rabatt in Euro
basis["umsatz"] = basis["unit_price"] * basis["product_quantity"]    # bezahlter Betrag
basis["monat"] = basis["date"].dt.to_period("M")

# Marke über die ersten 3 Zeichen der sku
basis["short"] = basis["sku"].str[:3]
basis = basis.merge(brands, on="short", how="left").rename(columns={"long": "marke"})

# Produktname dazuholen
basis = basis.merge(products_cl[["sku", "name"]], on="sku", how="left")

## Vorläufige Kategorien
Bis F3.1 fertig ist, teilen wir die Produkte über Stichwörter im Namen ein.
Alles in F4 läuft auf eigenen Kopien (`f4_produkte`, `f4_basis`) und alle Namen beginnen mit `f4_`. So überschreiben wir nichts aus F1–F3. Die Spalte heißt `kategorie_vorl`, damit sie nicht mit Miguels `kategorie` aus F3.1 kollidiert.

Die Reihenfolge der Regeln ist wichtig: Eine „Case for MacBook“ soll bei Hüllen landen, nicht bei Mac. Diese Einteilung ist ein erster Entwurf und wird durch F3.1 ersetzt.

In [ ]:
import re

# Reihenfolge = Priorität: die erste passende Regel gewinnt.
# Echte Apple-Geräte erkennen wir daran, dass der Name mit "Apple MacBook", "Apple iPhone" usw. beginnt.
# Danach kommt Zubehör, damit z. B. eine "Case for MacBook" nicht als Mac zählt.
f4_vorne = r"^(open - |\(open\) |like new - )?"
f4_regeln = [
    ("Service",                 r"\b(repair|replacement|installation|service)\b"),
    ("Mac",                     f4_vorne + r"apple (macbook|imac|mac)\b"),
    ("iPhone und iPad",         f4_vorne + r"apple (iphone|ipad)\b"),
    ("Apple Watch",             f4_vorne + r"(apple watch|watch apple)\b"),
    ("Hüllen und Taschen",      r"\b(case|cases|sleeve|cover|bag|backpack|briefcase|housing|funda|protector|skin|film|folio|shell)\b"),
    ("Armbänder",               r"\b(strap|band|armband|bracelet|correa|loop|belt)\b"),
    ("Kabel und Strom",         r"\b(cable|adapter|hub|docks?|docking|charger|charging|battery|powerbank|power bank|powerstation|plug)\b"),
    ("Speicher",                r"\b(ssd|hdd|hard drive|hard disk|drive|nas|raid|memory|ram|disk|storage|pendrive|jumpdrive|lacie|time capsule|microsd\w*|sdhc|sdxc|qnap|synology|drobo|my cloud|my passport)\b"),
    ("Audio",                   r"\b(headphones?|earphones?|speakers?|headset|beats|jbl|audio|earpods|airpods|sound)\b"),
    ("Monitore und Eingabe",    r"\b(monitor|display|keyboard|mouse|trackpad|pen|pointer|stylus|wacom)\b"),
    ("Netzwerk und Smart Home", r"\b(wi-fi|wifi|router|switch|repeater|extender|hue|smart|sensor|camera|drone|minidrone|thermometer|scale)\b"),
]

def f4_kategorie_finden(name):
    name = str(name).lower()
    for kategorie, muster in f4_regeln:
        if re.search(muster, name):
            return kategorie
    return "Sonstiges"

# Eigene Kopien für F4: so verändern wir die gemeinsamen Tabellen der anderen nicht
f4_produkte = products_cl.copy()
f4_basis = basis.copy()

f4_produkte["kategorie_vorl"] = f4_produkte["name"].apply(f4_kategorie_finden)
f4_basis["kategorie_vorl"] = f4_basis["name"].apply(f4_kategorie_finden)

# Kontrolle: Wie viele Produkte landen wo? "Sonstiges" sollte nicht zu groß sein.
f4_produkte["kategorie_vorl"].value_counts()

## F4.1 · Wie verteilen sich die Preise in den Kategorien?
Grundlage: der **Katalog** (`products_cl`), jedes Produkt einmal.
Die x-Achse ist logarithmisch, weil die Preise von 3 € bis über 15.000 € reichen. Sonst wären die günstigen Kategorien nur ein Strich.

In [ ]:
# Kategorien nach Median-Preis sortieren, damit man die Reihenfolge sofort sieht
f4_1_reihenfolge = (f4_produkte.groupby("kategorie_vorl")["price"]
                    .median().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
sns.boxplot(data=f4_produkte, x="price", y="kategorie_vorl",
            order=f4_1_reihenfolge, color="#7FA7D9", fliersize=2)
plt.xscale("log")
plt.xticks([10, 100, 1000, 10000], ["10 €", "100 €", "1.000 €", "10.000 €"])
plt.title("F4.1 · Katalogpreise je Kategorie (vorläufige Kategorien)")
plt.xlabel("Katalogpreis in € (logarithmisch)")
plt.ylabel("")
plt.tight_layout()
plt.show()

# Kennzahlen zum Diagramm
f4_1_tabelle = (f4_produkte.groupby("kategorie_vorl")["price"]
                .agg(produkte="count", median="median", minimum="min", maximum="max")
                .round(2).sort_values("median", ascending=False))
f4_1_tabelle

## F4.2 · Welche Preisklassen gibt es?
Wir teilen die Katalogpreise in fünf Klassen ein. Die Grenzen sind so gewählt, dass sie leicht zu merken sind und für Eniac Sinn ergeben: Zubehör liegt meist unter 50 €, Macs meist über 1.000 €.

In [ ]:
# Grenzen und Namen der Preisklassen
f4_grenzen = [0, 50, 100, 500, 1000, float("inf")]
f4_klassen = ["unter 50 €", "50–100 €", "100–500 €", "500–1.000 €", "über 1.000 €"]

f4_produkte["f4_preisklasse"] = pd.cut(f4_produkte["price"], bins=f4_grenzen, labels=f4_klassen)
f4_basis["f4_preisklasse"] = pd.cut(f4_basis["price"], bins=f4_grenzen, labels=f4_klassen)

# Wie viele Produkte gibt es pro Preisklasse im Katalog?
f4_2_katalog = f4_produkte["f4_preisklasse"].value_counts().reindex(f4_klassen)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=f4_2_katalog.index, y=f4_2_katalog.values, color="#7FA7D9")
ax.bar_label(ax.containers[0], labels=[f"{w:,}".replace(",", ".") for w in f4_2_katalog.values])
plt.title("F4.2 · Anzahl Produkte im Katalog je Preisklasse")
plt.xlabel("Preisklasse (Katalogpreis)")
plt.ylabel("Anzahl Produkte")
plt.tight_layout()
plt.show()

# Anteil in Prozent: Katalog vs. tatsächlich verkaufte Stück
f4_2_vergleich = pd.DataFrame({
    "anteil_katalog_%": (f4_produkte["f4_preisklasse"].value_counts(normalize=True) * 100),
    "anteil_verkaufte_stueck_%": (f4_basis.groupby("f4_preisklasse", observed=False)["product_quantity"].sum()
                                  / f4_basis["product_quantity"].sum() * 100),
}).reindex(f4_klassen).round(1)
f4_2_vergleich

## F4.3 · Welche Preisklasse bringt den meisten Umsatz?
Grundlage: die **Verkäufe** (`basis`). Umsatz = bezahlter Preis × Menge.

In [ ]:
f4_3_umsatz = f4_basis.groupby("f4_preisklasse", observed=False)["umsatz"].sum().reindex(f4_klassen)
f4_3_anteil = (f4_3_umsatz / f4_3_umsatz.sum() * 100).round(1)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=f4_3_umsatz.index, y=f4_3_umsatz.values / 1_000_000, color="#5BA58F")
ax.bar_label(ax.containers[0], labels=[f"{w:.1f} %".replace(".", ",") for w in f4_3_anteil])
plt.title("F4.3 · Umsatz je Preisklasse")
plt.xlabel("Preisklasse (Katalogpreis)")
plt.ylabel("Umsatz in Mio. €")
plt.tight_layout()
plt.show()

f4_3_tabelle = pd.DataFrame({
    "umsatz_eur": f4_3_umsatz.round(0),
    "anteil_umsatz_%": f4_3_anteil,
    "verkaufte_stueck": f4_basis.groupby("f4_preisklasse", observed=False)["product_quantity"].sum().reindex(f4_klassen),
})
f4_3_tabelle

## F4.4 · Werden teure Produkte stärker rabattiert als günstige?
Grundlage: die **Verkäufe** (`basis`). Wir vergleichen den durchschnittlichen Rabatt in Prozent je Preisklasse.
Prozent macht günstige und teure Produkte vergleichbar. In Euro wäre der Rabatt bei teuren Produkten automatisch größer.

In [ ]:
f4_4_tabelle = (f4_basis.groupby("f4_preisklasse", observed=False)
                .agg(rabatt_mittel_prozent=("rabatt", "mean"),
                     rabatt_median_prozent=("rabatt", "median"),
                     rabatt_mittel_eur=("rabatt_eur", "mean"))
                .reindex(f4_klassen))
f4_4_tabelle[["rabatt_mittel_prozent", "rabatt_median_prozent"]] *= 100
f4_4_tabelle = f4_4_tabelle.round(1)

plt.figure(figsize=(9, 5))
ax = sns.barplot(x=f4_4_tabelle.index, y=f4_4_tabelle["rabatt_mittel_prozent"], color="#D98C5F")
ax.bar_label(ax.containers[0], labels=[f"{w:.1f} %".replace(".", ",") for w in f4_4_tabelle["rabatt_mittel_prozent"]])
plt.title("F4.4 · Durchschnittlicher Rabatt je Preisklasse")
plt.xlabel("Preisklasse (Katalogpreis)")
plt.ylabel("Ø Rabatt in %")
plt.tight_layout()
plt.show()

f4_4_tabelle